# APEDP-X / CPMD: End-to-End Scientific Discovery & Epistemic Referee Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/owner/science/blob/main/notebooks/apedp_scientific_discovery_tutorial.ipynb)
[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/owner/science/blob/main/notebooks/apedp_scientific_discovery_tutorial.ipynb)

Welcome to **APEDP-X (Automated Physics & Epistemic Discovery Platform - eXtended)**.
This interactive tutorial demonstrates the complete scientific discovery pipeline governed by strict epistemic refereeing:

1. **Universal Environment & Storage Adapter** (`apedp_dyn.notebook_adapter`)
2. **Observational Data Simulation** (Microstructure orderflow arrivals & price impact)
3. **Candidate Hypothesis Generation** (`HawkesOrderFlowProposer` & `SparseMicrostructureProposer`)
4. **Stage 1-6 Epistemic Referee Adjudication** (Novelty, Diebold-Mariano incrementality, Deflated Sharpe Ratio)
5. **Anti-Snooping Holdout Discipline** (`FullLineageExperimentLedger` & slice burning)
6. **Remote Storage Synchronization & Immutable Snapshots** (`remote_storage`)


## Step 1: Universal Runtime Adapter Initialization
We initialize the runtime adapter to detect the platform (Google Colab, Kaggle, Jupyter, or Local Workstation), configure temporary scratch storage, and verify holdout persistence safeguards.

In [ ]:
import sys, os
from pathlib import Path

from apedp_dyn.notebook_adapter import init_environment, temporary_storage_scope

# Initialize environment (auto-detects Colab, Kaggle, Jupyter, or Local)
config = init_environment(
    mount_gdrive=False,              # Set True in Google Colab to preserve holdout ledgers across VM recycles
    temp_dir="/tmp/apedp_scratch",   # High-speed scratch storage for intermediate computations
    auto_compile_kernel=False        # Uses bit-accurate pure-Python fallback if Rust toolchain is absent
)

print(config.summary())
config  # Renders rich interactive HTML badge in Jupyter / Colab cells

## Step 2: Synthetic Observational Microstructure Data
We simulate high-frequency trade arrival point processes (Hawkes self-exciting intensity) and mid-price series for in-sample discovery, validation, and confirmation holdout adjudication.

In [ ]:
import numpy as np

np.random.seed(42)
n_steps = 2000
dt = 0.1

# Simulate self-exciting Hawkes order arrival intensities
base_intensity = 0.5
intensities = np.zeros(n_steps)
arrivals = np.zeros(n_steps)
curr_intensity = base_intensity

for t in range(n_steps):
    arrivals[t] = np.random.poisson(curr_intensity * dt)
    curr_intensity = base_intensity + 0.8 * (curr_intensity - base_intensity) + 0.5 * arrivals[t]
    intensities[t] = curr_intensity

# Simulate mid-price random walk with flow impact
noise = np.random.normal(0, 0.05, n_steps)
price_changes = 0.02 * (arrivals - np.mean(arrivals)) + noise
prices = 100.0 + np.cumsum(price_changes)

print(f"Generated {n_steps} microstructure steps. Final Price: {prices[-1]:.2f}, Mean Arrival Rate: {np.mean(arrivals):.2f}")

## Step 3: Candidate Hypothesis Generation (Proposers)
Per `AGENTS.md §1.1`, candidate hypotheses are strictly emitted by independent Proposers and never decide their own acceptance.

In [ ]:
import pandas as pd
from apedp_dyn.proposers.hawkes_orderflow_proposer import HawkesOrderFlowProposer
from apedp_dyn.proposers.sparse_microstructure_proposer import SparseMicrostructureProposer

hawkes_proposer = HawkesOrderFlowProposer(decay_beta=1.5, intensity_threshold_quantile=0.80)
micro_proposer = SparseMicrostructureProposer(l1_ratios=[0.5], alphas=[0.001])

# Prepare observational inputs for proposers
arrival_timestamps = np.cumsum(0.1 + np.abs(price_changes[:500]))
hawkes_candidates = hawkes_proposer.fit_and_propose(arrival_timestamps)

df_micro = pd.DataFrame({
    "price": prices[:500],
    "V": np.abs(arrivals[:500]) * 1000.0 + 100.0,
    "taker_buy": np.abs(arrivals[:500]) * 550.0 + 50.0,
})
micro_candidates = micro_proposer.fit_and_propose({"BTC": df_micro})

print(f"Hawkes Proposer emitted {len(hawkes_candidates)} candidate hypotheses:")
for c in hawkes_candidates[:2]:
    print(f"  • [{c.candidate_id}] Archetype: {c.strategy_archetype.value} | Formula: {c.formula}")

print(f"\nSparse Microstructure Proposer emitted {len(micro_candidates)} candidate hypotheses:")
for c in micro_candidates[:2]:
    print(f"  • [{c.candidate_id}] Archetype: {c.strategy_archetype.value} | Formula: {c.formula}")


## Step 4: Multi-Stage Epistemic Referee Adjudication (Gates 1–6)
We adjudicate the proposals across the canonical preregistered epistemic gates:
- **Gate 4**: Predictive incrementality via Diebold-Mariano test ($p < 0.01$)
- **Gate 5**: Deflated Sharpe Ratio (DSR $\ge 0.95$) accounting for selection bias across multiple trials

In [ ]:
from apedp_dyn.epistemic.novelty_adjudication import evaluate_predictive_incrementality
from apedp_dyn.epistemic.trial_ledger import compute_deflated_sharpe_ratio

# Gate 4: Diebold-Mariano Predictive Incrementality
y_true = price_changes[1000:1500]
y_base = y_true + 0.8 * np.random.randn(len(y_true))
y_cand = y_true + 0.2 * np.random.randn(len(y_true))

dm_res = evaluate_predictive_incrementality(y_true, y_cand, y_base, alpha=0.01)
print(f"Gate 4 (Diebold-Mariano): Stat={dm_res.dm_statistic:.3f}, p-value={dm_res.p_value:.4e} -> Verdict: {dm_res.verdict}")

# Gate 5: Deflated Sharpe Ratio under search multiplicity
dsr_result = compute_deflated_sharpe_ratio(
    observed_sharpe=1.65,
    n_trials=50,
    var_trials=0.15,
    n_bars=500
)
dsr = dsr_result["dsr"]
dsr_status = "PASS" if dsr >= 0.95 else "FAIL"
print(f"Gate 5 (Deflated Sharpe Ratio): DSR={dsr:.4f} -> {dsr_status}")


## Step 5: Anti-Snooping Holdout Discipline (`AGENTS.md §1.2`)
Confirmation holdouts are consumable, scarce resources. If a candidate hypothesis fails out-of-sample confirmation, that `(dataset_id, slice_id)` slice is permanently burned in `FullLineageExperimentLedger`. Any subsequent attempt to reuse that slice triggers `HoldoutContaminationError`.

In [ ]:
from apedp_dyn.epistemic.experiment_ledger import (
    FullLineageExperimentLedger,
    CandidateLifecycleState,
    DatasetRole,
    HoldoutContaminationError,
)

ledger = FullLineageExperimentLedger(
    persistence_path=config.get_ledger_path("tutorial_experiment_ledger.jsonl")
)

# Register Candidate Alpha
ledger.register_candidate(
    candidate_id="cand_alpha_01",
    proposer_id="hawkes_orderflow",
    formula="alpha_model(flow, dt)"
)

# Expose Candidate Alpha to Confirmation Holdout Slice
ledger.record_data_exposure(
    candidate_id="cand_alpha_01",
    dataset_id="microstructure_2026",
    slice_identifier="holdout_slice_Q3",
    role=DatasetRole.CONFIRMATION_HOLDOUT,
    outcome_status="FAILED"
)

# Candidate Alpha fails confirmation and is rejected
ledger.update_lifecycle_state(
    candidate_id="cand_alpha_01",
    new_state=CandidateLifecycleState.CONFIRMATION_REJECTED,
    rejection_reason="Holdout RMSE exceeded bounds"
)

print("Burned Holdout Registry:", ledger.get_burned_holdouts())

# Verify Anti-Snooping Invariant: Candidate Beta attempts to reuse the burned slice
ledger.register_candidate(
    candidate_id="cand_beta_02",
    proposer_id="microstructure",
    formula="beta_model(spread, vol)"
)

try:
    ledger.record_data_exposure(
        candidate_id="cand_beta_02",
        dataset_id="microstructure_2026",
        slice_identifier="holdout_slice_Q3",
        role=DatasetRole.CONFIRMATION_HOLDOUT
    )
    print("ERROR: Holdout contamination check failed to trigger!")
except HoldoutContaminationError as exc:
    print(f"\n[SUCCESS] Anti-snooping firewall triggered as expected:\n{exc}")

## Step 6: Scoped Scratch Workspaces & Immutable Snapshots
We demonstrate `temporary_storage_scope()` for fast, isolated intermediate data with guaranteed cleanup, and `export_snapshot()` for auditable persistence.

In [ ]:
# 1. Scoped temporary scratch workspace
with temporary_storage_scope() as scratch_dir:
    scratch_file = scratch_dir / "intermediate_sweep.dat"
    scratch_file.write_text("intermediate tensor sweep state")
    print(f"Scratch file active at: {scratch_file} (exists: {scratch_file.exists()})")

print(f"After exiting context, scratch directory is safely purged: {scratch_file.exists()}")

# 2. Export immutable timestamped ledger snapshot
snapshot_path = ledger.export_snapshot()
print(f"\nExported immutable ledger snapshot:\n  {snapshot_path}")